# Creating the parquet dataset from SQLite tables

In [1]:
import os
from pathlib import Path
import sys
node_type = os.getenv('BB_CPU')
venv_dir = f'/rds/homes/g/gaddcz/Projects/CPRD/virtual-envTorch2.0-{node_type}'
venv_site_pkgs = Path(venv_dir) / 'lib' / f'python{sys.version_info.major}.{sys.version_info.minor}' / 'site-packages'
if venv_site_pkgs.exists():
    sys.path.insert(0, str(venv_site_pkgs))
    print(f"Added path '{venv_site_pkgs}' at start of search paths.")
else:
    print(f"Path '{venv_site_pkgs}' not found. Check that it exists and/or that it exists for node-type '{node_type}'.")

!pwd

%load_ext autoreload
%autoreload 2

Added path '/rds/homes/g/gaddcz/Projects/CPRD/virtual-envTorch2.0-icelake/lib/python3.10/site-packages' at start of search paths.
/rds/homes/g/gaddcz/Projects/CPRD/examples/data/2_build_pre_training_dataset/stratified_dataset


In [2]:
import torch
from hydra import compose, initialize
from omegaconf import OmegaConf
import logging
import time
import pickle 

from FastEHR.dataloader import FoundationalDataModule
from FastEHR.database.collector import SQLiteDataCollector

from CPRD.examples.data.study_criteria import t2d_inclusion_method

torch.manual_seed(1337)

logging.basicConfig(level=logging.INFO)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# device = "cpu"    # if more informative debugging statements are needed
print(f"Using device: {device}.")


Using device: cuda.


In [5]:
# load the configuration file, override any settings 
with initialize(version_base=None, config_path="../../../modelling/SurvivEHR/confs", job_name="dataset_creation_notebook"):
    cfg = compose(config_name="config_CompetingRisk11M", overrides=[])

# Create new dataset 
cfg.data.path_to_ds = "/rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/FineTune_Hypertension/"
print(OmegaConf.to_yaml(cfg))

is_decoder: true
data:
  batch_size: 64
  unk_freq_threshold: 0.0
  min_workers: 12
  global_diagnoses: false
  repeating_events: true
  path_to_db: /rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/cprd.db
  path_to_ds: /rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/FineTune_Hypertension/
  meta_information_path: /rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/PreTrain/meta_information_QuantJenny.pickle
  subsample_training: null
experiment:
  type: pre-train
  project_name: SurvivEHR
  run_id: ${head.SurvLayer}PreTrain_small_${experiment.seed}
  fine_tune_id: null
  notes: null
  tags: null
  train: true
  test: true
  verbose: true
  seed: 1337
  log: true
  log_dir: /rds/projects/s/subramaa-mum-predict/CharlesGadd_Oxford/FoundationModelOutput/
  ckpt_dir: /rds/projects/s/subramaa-mum-predict/CharlesGadd_Oxford/FoundationModelOutput/checkpoints/
fine_tuning:
  fine_tune_outcomes: null
  custo

### We now want to divide this existing training cohort into sub-populations

In [6]:
save_path = "/rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/ByRegion/"

### Build the dataset for each of these collections of these general practices splits

In [9]:
# Build for each group 
authority_groups = [["North East"], 
                    ["London"]]

for auth_group in authority_groups:

    path_to_ds = save_path + f"PreTrain_{'_'.join(auth_group)}/"
    path_to_split = save_path + f'practice_id_splits_{"_".join(auth_group)}.pickle'

    os.makedirs(path_to_ds, exist_ok=True)
    for split_dir in ["train", "test", "val"]:
        os.makedirs(path_to_ds + f"split={split_dir}", exist_ok=True)
    
    dm = FoundationalDataModule(path_to_db=cfg.data.path_to_db,
                                path_to_ds=path_to_ds,
                                load=False,
                                include_diagnoses=True,
                                include_measurements=True,
                                drop_missing_data=False,
                                drop_empty_dynamic=True,
                                tokenizer="tabular",
                                overwrite_practice_ids=path_to_split,
                                overwrite_meta_information=cfg.data.meta_information_path,
                                num_threads=1
                               )

vocab_size = dm.train_set.tokenizer.vocab_size

print(f"{len(dm.train_set)} training patients")
print(f"{len(dm.val_set)} validation patients")
print(f"{len(dm.test_set)} test patients")
print(f"{vocab_size} vocab elements")

INFO:root:Building Polars datasets and saving to /rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/ByRegion/PreTrain_North East/
INFO:root:Using train/test/val splits from /rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/ByRegion/practice_id_splits_North East.pickle
INFO:root:Processing test split...
Thread generating parquet for 4 practices: 100%|██████████| 4/4 [03:17<00:00, 49.35s/it]
INFO:root:Created dataset at /rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/ByRegion/PreTrain_North East/split=test with 110628 number of samples
Getting file row counts. This allows the creation of an index to file map, increasing read efficiency: 444it [00:02, 152.63it/s]
INFO:root:	 Obtained with a total of 110628 samples
INFO:root:Processing train split...
Thread generating parquet for 50 practices:  10%|█         | 5/50 [12:31<1:52:45, 150.34s/it]

KeyboardInterrupt



In [ ]:
dm.train_set.view_sample(1, max_dynamic_events=None, report_time=True)